# Michaelis-Menten Reduction

This notebook shows two reductions for a Michaelis-Menten enzyme mechanism: first a conservation-law reduction, then a quasi-steady-state approximation (QSSA). The goal is to keep the substrate and product dynamics while eliminating intermediate enzyme-complex states.


## Model

The reaction structure is

$$
E + S \rightleftharpoons C \rightarrow E + P.
$$

With total enzyme $E_T$, the conservation law is

$$
E + C = E_T.
$$

After substituting $E = E_T - C$, the system is

$$
\begin{aligned}
\dot{S} &= -k_1(E_T-C)S + k_2C, \\
\dot{C} &= k_1(E_T-C)S - (k_2+k_3)C, \\
\dot{P} &= k_3C.
\end{aligned}
$$


In [ ]:
import numpy as np
from sympy import Symbol, simplify

from autoreduce.system.system import System
from autoreduce.utils.reduction import get_reducible

S = Symbol("S")
C = Symbol("C")
P = Symbol("P")
k1 = Symbol("k1")
k2 = Symbol("k2")
k3 = Symbol("k3")
E_total = Symbol("E_total")

E = E_total - C
x = [S, C, P]
f = [
    -k1 * E * S + k2 * C,
    k1 * E * S - (k2 + k3) * C,
    k3 * C,
]

system = System(
    x,
    f,
    params=[k1, k2, k3, E_total],
    params_values=[1.0, 0.5, 0.25, 1.0],
    x_init=[10.0, 0.0, 0.0],
    C=np.array([[1, 0, 0], [0, 0, 1]]),
)


## QSSA Reduction

For the standard Michaelis-Menten reduction, the enzyme complex $C$ is treated as the fast state. AutoReduce solves $\dot{C}=0$ and substitutes the resulting expression for $C$ into the slow dynamics.


In [ ]:
reducible_system = get_reducible(system)
reduced_system, collapsed_system = reducible_system.solve_timescale_separation(
    [S, P],
    fast_states=[C],
)

[simplify(expr) for expr in reduced_system.f]


## Reduced Dynamics

The reduced system has states $[S, P]$. The symbolic dynamics are

$$
\begin{aligned}
\dot{S} &= -\frac{E_T S k_1 k_3}{S k_1 + k_2 + k_3}, \\
\dot{P} &= \frac{E_T S k_1 k_3}{S k_1 + k_2 + k_3}.
\end{aligned}
$$

The collapsed system records the eliminated fast state and its algebraic condition.


In [ ]:
collapsed_system.x, collapsed_system.f
